# KisanCredit Profitability Model - Comprehensive Evaluation

This notebook provides a detailed analysis of the trained profitability scoring model.

## Contents
1. Load Model and Test Data
2. Model Performance Metrics
3. Feature Importance Analysis
4. SHAP (SHapley Additive exPlanations) Analysis
5. Prediction Distribution Analysis
6. Error Analysis
7. Sample Predictions with Explanations

---

In [ ]:
# Imports
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (
    mean_squared_error, 
    mean_absolute_error, 
    r2_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("Imports successful!")

## 1. Load Model and Test Data

In [ ]:
# Load trained model
model_path = project_root / "models" / "profitability_model_latest.pkl"
model_data = joblib.load(model_path)

model = model_data['model']
scaler = model_data['scaler']
feature_names = model_data['feature_names']
metadata = model_data.get('metadata', {})

print(f"Model loaded: {model.__class__.__name__}")
print(f"Number of features: {len(feature_names)}")
print(f"\nMetadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")

In [ ]:
# Generate test data
from src.pipeline.data_generator import SyntheticDataGenerator
from src.features import FeatureEngineeringPipeline

print("Generating test dataset (1000 samples)...")
generator = SyntheticDataGenerator()
applications_df = generator.generate_dataset(1000)

# Extract features
pipeline = FeatureEngineeringPipeline()
applications = applications_df.to_dict('records')
features_df = pipeline.extract_batch(applications)

# Generate target (same formula as training)
def generate_target(df):
    score = 0.0
    if 'income_consistency_score' in df.columns:
        score += 0.40 * df['income_consistency_score'].fillna(0)
    if 'expense_to_income_ratio' in df.columns:
        score += 0.25 * (1 - df['expense_to_income_ratio'].clip(0, 1).fillna(0.5))
    if 'social_network_strength' in df.columns:
        score += 0.15 * df['social_network_strength'].fillna(0)
    if 'discipline_overall_score' in df.columns:
        score += 0.10 * df['discipline_overall_score'].fillna(0)
    if 'behavioral_risk_score' in df.columns:
        score += 0.10 * (1 - df['behavioral_risk_score'].clip(0, 1).fillna(0))
    noise = np.random.normal(0, 0.05, size=len(df))
    return (score + noise).clip(0, 1)

features_df['profitability_score'] = generate_target(features_df)

# Split features and target
X_test = features_df[feature_names]
y_test = features_df['profitability_score']

print(f"Test set: {len(X_test)} samples, {len(feature_names)} features")
print(f"Target distribution: mean={y_test.mean():.3f}, std={y_test.std():.3f}")

## 2. Model Performance Metrics

In [ ]:
# Make predictions
X_test_scaled = scaler.transform(X_test)
y_pred = model.predict(X_test_scaled)

# Regression metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("=" * 60)
print("REGRESSION METRICS")
print("=" * 60)
print(f"R² Score:          {r2:.4f}")
print(f"RMSE:              {rmse:.4f}")
print(f"MAE:               {mae:.4f}")
print(f"MAPE:              {mape:.2f}%")

# Classification metrics (threshold = 0.6)
threshold = 0.6
y_pred_binary = (y_pred >= threshold).astype(int)
y_test_binary = (y_test >= threshold).astype(int)

precision = precision_score(y_test_binary, y_pred_binary)
recall = recall_score(y_test_binary, y_pred_binary)
f1 = f1_score(y_test_binary, y_pred_binary)
accuracy = (y_pred_binary == y_test_binary).mean()

print("\n" + "=" * 60)
print(f"CLASSIFICATION METRICS (threshold={threshold})")
print("=" * 60)
print(f"Accuracy:          {accuracy:.4f}")
print(f"Precision:         {precision:.4f}")
print(f"Recall:            {recall:.4f}")
print(f"F1 Score:          {f1:.4f}")

# Approval rate
approval_rate = y_pred_binary.mean()
print(f"\nApproval Rate:     {approval_rate:.2%}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test_binary, y_pred_binary)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Reject', 'Approve'],
            yticklabels=['Reject', 'Approve'],
            ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix (threshold={threshold})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Classification report
print("\nDetailed Classification Report:")
print(classification_report(y_test_binary, y_pred_binary, 
                          target_names=['Reject', 'Approve']))

## 3. Feature Importance Analysis

In [ ]:
# Get feature importance
importance = model.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

# Top 20 features
top_20 = feature_importance_df.head(20)

fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(data=top_20, y='feature', x='importance', palette='viridis', ax=ax)
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
ax.set_title('Top 20 Most Important Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 10 Most Important Features:")
print(feature_importance_df.head(10).to_string(index=False))

In [ ]:
# Feature importance by category
categories = {
    'Income': ['income_', 'credit_'],
    'Expense': ['expense_', 'debit_'],
    'Social': ['social_', 'contact_'],
    'Location': ['location_'],
    'Discipline': ['discipline_', 'late_payment'],
    'Behavioral': ['behavioral_', 'app_usage']
}

category_importance = {}
for category, prefixes in categories.items():
    mask = feature_importance_df['feature'].str.contains('|'.join(prefixes))
    category_importance[category] = feature_importance_df[mask]['importance'].sum()

# Plot category importance
fig, ax = plt.subplots(figsize=(10, 6))
categories_sorted = sorted(category_importance.items(), key=lambda x: x[1], reverse=True)
cats, imps = zip(*categories_sorted)
sns.barplot(x=list(imps), y=list(cats), palette='coolwarm', ax=ax)
ax.set_xlabel('Total Importance', fontsize=12)
ax.set_ylabel('Feature Category', fontsize=12)
ax.set_title('Feature Importance by Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. SHAP Analysis

SHAP (SHapley Additive exPlanations) provides model interpretability by showing how each feature contributes to individual predictions.

In [ ]:
import shap

# Create SHAP explainer
print("Creating SHAP explainer (this may take a minute)...")
explainer = shap.TreeExplainer(model)

# Calculate SHAP values for test set (sample 100 for speed)
sample_size = min(100, len(X_test_scaled))
X_sample = X_test_scaled[:sample_size]
shap_values = explainer.shap_values(X_sample)

print(f"SHAP values calculated for {sample_size} samples")

In [ ]:
# SHAP Summary Plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False)
plt.title('SHAP Feature Importance Summary', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot (mean absolute SHAP values)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, 
                 plot_type="bar", show=False)
plt.title('Mean Absolute SHAP Values', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 5. Prediction Distribution Analysis

In [ ]:
# Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter plot
axes[0].scatter(y_test, y_pred, alpha=0.5, s=20)
axes[0].plot([0, 1], [0, 1], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Profitability Score', fontsize=12)
axes[0].set_ylabel('Predicted Profitability Score', fontsize=12)
axes[0].set_title('Predicted vs Actual Scores', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribution comparison
axes[1].hist(y_test, bins=30, alpha=0.5, label='Actual', color='blue')
axes[1].hist(y_pred, bins=30, alpha=0.5, label='Predicted', color='orange')
axes[1].axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold={threshold}')
axes[1].set_xlabel('Profitability Score', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Score Distribution', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Prediction error distribution
errors = y_pred - y_test

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Error histogram
axes[0].hist(errors, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Prediction Error (Predicted - Actual)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Prediction Error Distribution', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Residual plot
axes[1].scatter(y_pred, errors, alpha=0.5, s=20)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Score', fontsize=12)
axes[1].set_ylabel('Residual (Predicted - Actual)', fontsize=12)
axes[1].set_title('Residual Plot', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Error Statistics:")
print(f"  Mean Error:     {errors.mean():.4f}")
print(f"  Std Error:      {errors.std():.4f}")
print(f"  Min Error:      {errors.min():.4f}")
print(f"  Max Error:      {errors.max():.4f}")

## 6. Error Analysis

In [ ]:
# Find worst predictions
error_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_pred,
    'error': np.abs(errors),
    'actual_decision': y_test_binary.values,
    'predicted_decision': y_pred_binary
})

# Worst 10 predictions
worst_10 = error_df.nlargest(10, 'error')

print("Top 10 Worst Predictions:")
print(worst_10.to_string(index=False))

# Misclassifications
misclassified = error_df[error_df['actual_decision'] != error_df['predicted_decision']]
print(f"\nTotal Misclassifications: {len(misclassified)} ({len(misclassified)/len(error_df)*100:.2f}%)")

if len(misclassified) > 0:
    print("\nSample Misclassifications:")
    print(misclassified.head(5).to_string(index=False))

## 7. Sample Predictions with Explanations

In [ ]:
# Explain 3 sample predictions
from src.models import ModelExplainer

explainer_obj = ModelExplainer(str(model_path))

# Select diverse samples: low, medium, high scores
low_idx = y_pred.argmin()
high_idx = y_pred.argmax()
mid_idx = np.abs(y_pred - 0.5).argmin()

samples = [low_idx, mid_idx, high_idx]
labels = ['Lowest Score', 'Medium Score', 'Highest Score']

for idx, label in zip(samples, labels):
    sample_features = X_test.iloc[idx:idx+1]
    explanation = explainer_obj.explain_prediction(sample_features, top_n=5)
    
    print("=" * 80)
    print(f"{label.upper()} - Sample {idx}")
    print("=" * 80)
    print(f"Predicted Score:   {explanation['prediction']:.4f}")
    print(f"Decision:          {explanation['decision']}")
    print(f"Base Value:        {explanation['base_value']:.4f}")
    print(f"\nTop 5 Contributing Features:")
    
    for i, contrib in enumerate(explanation['top_contributions'], 1):
        feature = contrib['feature']
        value = contrib['value']
        shap_val = contrib['shap_value']
        direction = "↑" if shap_val > 0 else "↓"
        print(f"  {i}. {feature:40s} = {value:8.4f}  {direction} {shap_val:+.4f}")
    
    print()

In [ ]:
# SHAP waterfall plot for one sample
sample_idx = high_idx
sample_shap = shap.Explanation(
    values=shap_values[sample_idx] if sample_idx < len(shap_values) else shap_values[0],
    base_values=explainer.expected_value,
    data=X_sample[sample_idx] if sample_idx < len(X_sample) else X_sample[0],
    feature_names=feature_names
)

print(f"\nSHAP Waterfall Plot for Sample {sample_idx}:")
shap.waterfall_plot(sample_shap, max_display=15, show=False)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrates:
- **Model Performance**: Comprehensive regression and classification metrics
- **Feature Analysis**: Importance rankings and category-level insights
- **Explainability**: SHAP values for model interpretability
- **Error Analysis**: Understanding prediction mistakes
- **Sample Explanations**: Detailed breakdowns of individual predictions

The model achieves strong performance on the test set and provides interpretable predictions through SHAP analysis.